# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

[FAIR² dataset on sen.science](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

### List of available record sets and their `@id` fields

In [ ]:
# Get all record sets and show their @id and name
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets defined in the metadata.")
else:
    for rs in record_sets:
        print(f"@id: {rs['@id']} \t name: {rs.get('name', '')}")

### Explore fields and columns within each record set
In this dataset, let's enumerate fields, columns, and their `@id` for each record set.

Note: If there are multiple record sets, select one for further work below.

In [ ]:
# List field @id's and columns for each record set
for rs in dataset.record_sets:
    print(f"\nRecordSet: {rs['@id']} ({rs.get('name', '')})")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            f_id = field.get('@id', field) if isinstance(field, dict) else field
            print(f"  Field @id: {f_id}")
            # List columns if this field contains columns
            if isinstance(field, dict) and 'column' in field:
                columns = field['column'] if isinstance(field['column'], list) else [field['column']]
                for col in columns:
                    c_id = col.get('@id', col) if isinstance(col, dict) else col
                    print(f"    Column @id: {c_id}")
    else:
        print("  No fields present.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

First, let's collect the list of available record set `@id`s, then extract one (or all).

In [ ]:
# Gather the available record set @ids
record_sets_list = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}

for record_set_id in record_sets_list:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records from RecordSet {record_set_id}.")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Pick the main/first record set for continued analysis
main_record_set_id = record_sets_list[0] if record_sets_list else None
if main_record_set_id and main_record_set_id in dataframes:
    print("\nColumns in main DataFrame:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: e.g., filtering records based on specific criteria, normalizing numeric fields, categorizing data, removing outliers, grouping, etc.

Below: Select a numeric field (using its `@id` from the schema) and a group field for demonstration.

> **Replace the field `@id`s below with actual values from the output above if needed.**

In [ ]:
# Set your field and group ids based on column names from previous cell!
df = dataframes[main_record_set_id]
# Example column names - please adjust as per real dataset fields!
# Let's try to find a numeric field, e.g., 'age_at_second_crc', and a group field, e.g., 'sex'
import numpy as np

# Sample field names (replace with actual column names if different):
numeric_field = None
group_field = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field = col
    if col.lower() in {'sex', 'gender', 'anatomical_location', 'msi_status'}:
        group_field = col

print(f"Selected numeric field: {numeric_field}")
print(f"Selected group field: {group_field}")

# Filtering: for age > 50 as an example
if numeric_field and numeric_field in df.columns:
    threshold = 50
    filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping: by group_field
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
    else:
        print(f"Group field {group_field} not found.")
else:
    print('No suitable numeric field found.')

## 5. Visualization
Visualize data distributions or relationships between fields.

Here, we show an age histogram and group breakdown.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].astype(float), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
if group_field and numeric_field and group_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field].astype(float))
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

- Successfully loaded clinical and pathological data on second primary colorectal cancer using `mlcroissant`.
- Demonstrated selection and overview of record sets and fields by their `@id` in the Croissant schema.
- Extracted a main record set as a DataFrame, performed numeric filtering, normalization, and categorical grouping using field `@id`s.
- Visualized key field distributions and categories.

This demonstrates how to interactively explore, filter, and analyze FAIR data described by a Croissant schema. You may now proceed to domain-specific analyses, modeling, or integrate additional Croissant-based datasets as needed.